In [7]:
import requests
import pandas as pd
import json
import time
from datetime import datetime, timezone
from pathlib import Path
import os

if os.path.exists("/content"):
    PASTA_DADOS = Path("/content/dados_cripto")
else:
    PASTA_DADOS = Path("./dados_cripto")

PASTA_DADOS.mkdir(parents=True, exist_ok=True)

def agora_utc():
    return datetime.now(timezone.utc).isoformat()

def consultar_json(url, params=None, metodo="GET", payload=None):
    """
    Consulta uma API e devolve o JSON.
    Interrompe com uma mensagem clara se houver erro.
    """
    resposta = requests.request(
        method=metodo,
        url=url,
        params=params,
        json=payload,
        timeout=30,
        headers={"User-Agent": "datathon-cripto-pesquisa/1.0"}
    )

    resposta.raise_for_status()
    return resposta.json()

def salvar_csv(registros, nome_arquivo):
    df = pd.DataFrame(registros)
    caminho = PASTA_DADOS / nome_arquivo
    df.to_csv(caminho, index=False)
    print(f"Arquivo salvo: {caminho} | Registros: {len(df)}")
    return df

print("Pasta de dados:", PASTA_DADOS)
print("Horário UTC:", agora_utc())

Pasta de dados: dados_cripto
Horário UTC: 2026-11-02T20:38:44.695005+00:00


In [3]:
URL_BTC = "https://mempool.space/api"

mempool_btc = consultar_json(
    f"{URL_BTC}/mempool"
)

taxas_btc = consultar_json(
    f"{URL_BTC}/v1/fees/recommended"
)

print("=== MEMPOOL BITCOIN ===")
print(json.dumps(mempool_btc, indent=2, ensure_ascii=False))

print("\n=== TAXAS RECOMENDADAS ===")
print(json.dumps(taxas_btc, indent=2, ensure_ascii=False))

=== MEMPOOL BITCOIN ===
{
  "count": 50178,
  "vsize": 22418931,
  "total_fee": 5586189,
  "fee_histogram": [
    [
      5.794872,
      50248
    ],
    [
      4.922727,
      50033
    ],
    [
      3.62257,
      58620
    ],
    [
      3.0183487,
      50063
    ],
    [
      2.5285714,
      50039
    ],
    [
      2.1886232,
      88706
    ],
    [
      2.0089126,
      50846
    ],
    [
      1.1878629,
      53454
    ],
    [
      1.0091743,
      65128
    ],
    [
      1.0071429,
      71869
    ],
    [
      1.0057908,
      52814
    ],
    [
      1.0045046,
      50381
    ],
    [
      1.0004035,
      51302
    ],
    [
      1.0000578,
      72765
    ],
    [
      1.0000544,
      73539
    ],
    [
      1.0000408,
      167479
    ],
    [
      1.0000272,
      182203
    ],
    [
      1.0000136,
      73542
    ],
    [
      1.0,
      131094
    ],
    [
      0.961975,
      58707
    ],
    [
      0.71973956,
      57315
    ],
    [
      0.6

In [4]:
URL_ETH = "https://ethereum-rpc.publicnode.com"

def consultar_eth(metodo, parametros=None):
    return consultar_json(
        URL_ETH,
        metodo="POST",
        payload={
            "jsonrpc": "2.0",
            "method": metodo,
            "params": parametros or [],
            "id": 1
        }
    )

resultado_bloco = consultar_eth("eth_blockNumber")
resultado_taxa = consultar_eth("eth_gasPrice")
resultado_historico = consultar_eth(
    "eth_feeHistory",
    ["0x5", "latest", [25, 50, 75]]
)

for nome, resultado in [
    ("Bloco atual", resultado_bloco),
    ("Preço do gas", resultado_taxa),
    ("Histórico de taxas", resultado_historico)
]:
    print(f"\n=== {nome} ===")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))


=== Bloco atual ===
{
  "jsonrpc": "2.0",
  "result": "0x191c0f9",
  "id": 1
}

=== Preço do gas ===
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": "0xc00ad8c"
}

=== Histórico de taxas ===
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "baseFeePerGas": [
      "0xaf25c19",
      "0xb4fffc8",
      "0xb523b31",
      "0xbeae32d",
      "0xc00a51b",
      "0xc19d551"
    ],
    "gasUsedRatio": [
      0.6336579333333333,
      0.5030829772967629,
      0.7106887584876539,
      0.5285261833333333,
      0.532790420939061
    ],
    "baseFeePerBlobGas": [
      "0xa631f9",
      "0xa631f9",
      "0xa631f9",
      "0xaf207f",
      "0xb26e8e",
      "0xb471d6"
    ],
    "blobGasUsedRatio": [
      0.0,
      0.0,
      0.6666666666666666,
      0.23809523809523808,
      0.14285714285714285
    ],
    "oldestBlock": "0x191c0f5",
    "reward": [
      [
        "0x457d4a",
        "0x5f5e100",
        "0x30a86de7"
      ],
      [
        "0x7e78f6",
        "0xca8d1fe",
        "0x72

In [5]:
URL_SOL = "https://api.mainnet-beta.solana.com"

resultado_sol = consultar_json(
    URL_SOL,
    metodo="POST",
    payload={
        "jsonrpc": "2.0",
        "id": 1,
        "method": "getRecentPrioritizationFees",
        "params": []
    }
)

if "error" in resultado_sol:
    raise RuntimeError(resultado_sol["error"])

taxas_sol = resultado_sol["result"]

print("Quantidade de observações:", len(taxas_sol))
print("Primeiras observações:")
print(json.dumps(taxas_sol[:5], indent=2))

Quantidade de observações: 150
Primeiras observações:
[
  {
    "prioritizationFee": 0,
    "slot": 464621226
  },
  {
    "prioritizationFee": 0,
    "slot": 464621227
  },
  {
    "prioritizationFee": 0,
    "slot": 464621228
  },
  {
    "prioritizationFee": 0,
    "slot": 464621229
  },
  {
    "prioritizationFee": 0,
    "slot": 464621230
  }
]


In [ ]:
import pandas as pd
from pathlib import Path

URL_FNG = "https://api.alternative.me/fng/"
ARQUIVO_FNG = Path("./dados_cripto") / "fear_greed_historico.csv"

# Solicitar todo o histórico disponível
resultado_fng = consultar_json(
    URL_FNG,
    params={
        "limit": 0,
        "format": "json"
    }
)

dados_fng = resultado_fng["data"]

# Transformar em DataFrame
df_fng = pd.DataFrame(dados_fng)

# Converter campos numéricos
df_fng["value"] = pd.to_numeric(df_fng["value"])
df_fng["timestamp"] = pd.to_numeric(df_fng["timestamp"])

# Converter timestamp Unix para UTC
df_fng["timestamp_utc"] = pd.to_datetime(
    df_fng["timestamp"],
    unit="s",
    utc=True
)

# Se já existe histórico, preservar os registros anteriores
if ARQUIVO_FNG.exists():

    historico = pd.read_csv(ARQUIVO_FNG)

    combinado = pd.concat(
        [historico, df_fng],
        ignore_index=True
    )

else:
    combinado = df_fng.copy()

# Remover duplicatas pela identificação temporal
combinado["timestamp"] = pd.to_numeric(
    combinado["timestamp"]
)

combinado = combinado.drop_duplicates(
    subset=["timestamp"],
    keep="last"
)

# Ordenar cronologicamente
combinado = combinado.sort_values("timestamp")

# Salvar
combinado.to_csv(
    ARQUIVO_FNG,
    index=False
)

print("Total de observações:", len(combinado))

print(
    "Primeira observação:",
    combinado["timestamp_utc"].min()
)

print(
    "Última observação:",
    combinado["timestamp_utc"].max()
)

display(combinado.head(5))
display(combinado.tail(5))

Total de observações: 3193
Primeira observação: 2018-02-01 00:00:00+00:00
Última observação: 2026-11-02 00:00:00+00:00


,value,value_classification,timestamp,time_until_update,timestamp_utc
3222,30,Fear,1517443200,NaN,2018-02-01 00:00:00+00:00
3221,15,Extreme Fear,1517529600,NaN,2018-02-02 00:00:00+00:00
3220,40,Fear,1517616000,NaN,2018-02-03 00:00:00+00:00
3219,24,Extreme Fear,1517702400,NaN,2018-02-04 00:00:00+00:00
3218,11,Extreme Fear,1517788800,NaN,2018-02-05 00:00:00+00:00


,value,value_classification,timestamp,time_until_update,timestamp_utc
34,51,Neutral,1793232000,NaN,2026-10-29 00:00:00+00:00
33,30,Fear,1793318400,NaN,2026-10-30 00:00:00+00:00
32,30,Fear,1793404800,NaN,2026-10-31 00:00:00+00:00
31,34,Fear,1793491200,NaN,2026-11-01 00:00:00+00:00
30,37,Fear,1793577600,12070,2026-11-02 00:00:00+00:00


: 

In [7]:
URL_BINANCE = "https://api.binance.com/api/v3/klines"

def coletar_candles(symbol, limite=48):
    resposta = consultar_json(
        URL_BINANCE,
        params={
            "symbol": symbol,
            "interval": "1h",
            "limit": limite
        }
    )

    colunas = [
        "timestamp_abertura",
        "abertura",
        "maxima",
        "minima",
        "fechamento",
        "volume",
        "timestamp_fechamento",
        "volume_quote",
        "numero_trades",
        "volume_taker_base",
        "volume_taker_quote",
        "campo_ignorado"
    ]

    df = pd.DataFrame(resposta, columns=colunas)

    df["timestamp_abertura"] = pd.to_datetime(
        df["timestamp_abertura"], unit="ms", utc=True
    )

    df["timestamp_fechamento"] = pd.to_datetime(
        df["timestamp_fechamento"], unit="ms", utc=True
    )

    colunas_numericas = [
        "abertura", "maxima", "minima", "fechamento",
        "volume", "volume_quote", "numero_trades",
        "volume_taker_base", "volume_taker_quote"
    ]

    for coluna in colunas_numericas:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    df["ativo"] = symbol.replace("USDT", "")
    df["fonte"] = "Binance Spot"

    return df

candles = []

for simbolo in ["BTCUSDT", "ETHUSDT", "SOLUSDT"]:
    try:
        df = coletar_candles(simbolo, limite=48)
        candles.append(df)
        print(f"{simbolo}: {len(df)} candles coletados")
        time.sleep(0.3)
    except Exception as erro:
        print(f"Erro ao coletar {simbolo}: {erro}")

if candles:
    df_novos_candles = pd.concat(candles, ignore_index=True)
    caminho_mercado = PASTA_DADOS / "mercado_horario.csv"
    if caminho_mercado.exists():
        df_antigo = pd.read_csv(caminho_mercado)
        df_antigo["timestamp_abertura"] = pd.to_datetime(df_antigo["timestamp_abertura"], utc=True)
        df_antigo["timestamp_fechamento"] = pd.to_datetime(df_antigo["timestamp_fechamento"], utc=True)
        df_mercado = pd.concat([df_antigo, df_novos_candles], ignore_index=True)
        df_mercado = df_mercado.drop_duplicates(subset=["ativo", "timestamp_abertura"], keep="last").sort_values(["ativo", "timestamp_abertura"]).reset_index(drop=True)
    else:
        df_mercado = df_novos_candles
    df_mercado.to_csv(caminho_mercado, index=False)
    display(df_mercado.head())
else:
    raise RuntimeError(
        "Nenhum candle foi coletado. Verifique a conexão e a API."
    )

BTCUSDT: 48 candles coletados
ETHUSDT: 48 candles coletados
SOLUSDT: 48 candles coletados


,timestamp_abertura,abertura,maxima,minima,fechamento,volume,timestamp_fechamento,volume_quote,numero_trades,volume_taker_base,volume_taker_quote,campo_ignorado,ativo,fonte
0,2026-10-31 20:00:00+00:00,77588.83,77647.78,77562.00,77565.87,129.50384,2026-10-31 20:59:59.999000+00:00,1.004905e+07,19427,59.20290,4.593909e+06,0,BTC,Binance Spot
1,2026-10-31 21:00:00+00:00,77565.87,77682.01,77450.00,77682.01,152.79639,2026-10-31 21:59:59.999000+00:00,1.184571e+07,35557,77.73041,6.026077e+06,0,BTC,Binance Spot
2,2026-10-31 22:00:00+00:00,77682.00,77764.01,77537.83,77548.01,135.50779,2026-10-31 22:59:59.999000+00:00,1.052385e+07,39876,73.31559,5.693973e+06,0,BTC,Binance Spot
3,2026-10-31 23:00:00+00:00,77548.01,77596.26,77515.30,77582.02,93.82194,2026-10-31 23:59:59.999000+00:00,7.275792e+06,24982,51.27606,3.976267e+06,0,BTC,Binance Spot
4,2026-11-01 00:00:00+00:00,77582.02,77715.50,77517.20,77690.41,155.06017,2026-11-01 00:59:59.999000+00:00,1.203425e+07,37826,93.99385,7.294766e+06,0,BTC,Binance Spot


In [8]:
print("Registros por ativo:")
print(df_mercado.groupby("ativo").size())

print("\nIntervalo temporal por ativo:")
print(
    df_mercado.groupby("ativo")["timestamp_abertura"]
    .agg(["min", "max"])
)

print("\nValores ausentes:")
print(df_mercado.isna().sum())

print("\nCandles duplicados:")
print(
    df_mercado.duplicated(
        subset=["ativo", "timestamp_abertura"]
    ).sum()
)

print("\nPreços inválidos:")
print(
    (df_mercado["fechamento"] <= 0).sum()
)

Registros por ativo:
ativo
BTC    48
ETH    48
SOL    48
dtype: int64

Intervalo temporal por ativo:
                            min                       max
ativo                                                    
BTC   2026-10-31 20:00:00+00:00 2026-11-02 19:00:00+00:00
ETH   2026-10-31 20:00:00+00:00 2026-11-02 19:00:00+00:00
SOL   2026-10-31 20:00:00+00:00 2026-11-02 19:00:00+00:00

Valores ausentes:
timestamp_abertura      0
abertura                0
maxima                  0
minima                  0
fechamento              0
volume                  0
timestamp_fechamento    0
volume_quote            0
numero_trades           0
volume_taker_base       0
volume_taker_quote      0
campo_ignorado          0
ativo                   0
fonte                   0
dtype: int64

Candles duplicados:
0

Preços inválidos:
0


In [9]:
registro_btc = {
    "timestamp_coleta_utc": agora_utc(),
    "mempool_transacoes": mempool_btc.get("count"),
    "mempool_vsize_vbytes": mempool_btc.get("vsize"),
    "mempool_taxas_sats": mempool_btc.get("total_fee"),
    "taxa_fastest_sat_vb": taxas_btc.get("fastestFee"),
    "taxa_30min_sat_vb": taxas_btc.get("halfHourFee"),
    "taxa_1h_sat_vb": taxas_btc.get("hourFee"),
    "taxa_economica_sat_vb": taxas_btc.get("economyFee"),
    "taxa_minima_sat_vb": taxas_btc.get("minimumFee"),
}

df_btc = salvar_csv([registro_btc], "bitcoin_onchain.csv")
display(df_btc)

Arquivo salvo: dados_cripto/bitcoin_onchain.csv | Registros: 1


,timestamp_coleta_utc,mempool_transacoes,mempool_vsize_vbytes,mempool_taxas_sats,taxa_fastest_sat_vb,taxa_30min_sat_vb,taxa_1h_sat_vb,taxa_economica_sat_vb,taxa_minima_sat_vb
0,2026-11-02T20:01:19.910039+00:00,50178,22418931,5586189,2,1,1,1,1


In [10]:
historico_eth = resultado_historico.get("result", {})

def hex_para_int(valor):
    return int(valor, 16) if valor else None

def hex_para_gwei(valor):
    return hex_para_int(valor) / 1e9 if valor else None

taxas_base = historico_eth.get("baseFeePerGas", [])
recompensas = historico_eth.get("reward", [])
utilizacao = historico_eth.get("gasUsedRatio", [])

ultima_recompensa = recompensas[-1] if recompensas else []

registro_eth = {
    "timestamp_coleta_utc": agora_utc(),
    "bloco_atual": hex_para_int(resultado_bloco["result"]),
    "gas_price_gwei": hex_para_gwei(resultado_taxa["result"]),
    "base_fee_ultimo_bloco_gwei": (
        hex_para_gwei(taxas_base[-2])
        if len(taxas_base) >= 2 else None
    ),
    "base_fee_proximo_bloco_gwei": (
        hex_para_gwei(taxas_base[-1])
        if taxas_base else None
    ),
    "priority_fee_p50_gwei": (
        hex_para_gwei(ultima_recompensa[1])
        if len(ultima_recompensa) >= 2 else None
    ),
    "gas_utilizado_ultimo_bloco": (
        utilizacao[-1] if utilizacao else None
    )
}

df_eth = salvar_csv([registro_eth], "ethereum_onchain.csv")
display(df_eth)

Arquivo salvo: dados_cripto/ethereum_onchain.csv | Registros: 1


,timestamp_coleta_utc,bloco_atual,gas_price_gwei,base_fee_ultimo_bloco_gwei,base_fee_proximo_bloco_gwei,priority_fee_p50_gwei,gas_utilizado_ultimo_bloco
0,2026-11-02T20:01:38.136070+00:00,26329337,0.201371,0.201369,0.20302,0.007331,0.53279


In [11]:
timestamp_sol = agora_utc()

df_sol = pd.DataFrame(taxas_sol)

if not df_sol.empty:
    df_sol["timestamp_coleta_utc"] = timestamp_sol

    df_sol["slot"] = pd.to_numeric(
        df_sol["slot"], errors="coerce"
    )

    df_sol["prioritizationFee"] = pd.to_numeric(
        df_sol["prioritizationFee"], errors="coerce"
    )

    df_sol.to_csv(
        PASTA_DADOS / "solana_taxas_prioridade.csv",
        index=False
    )

    resumo_sol = {
        "timestamp_coleta_utc": timestamp_sol,
        "observacoes": len(df_sol),
        "taxa_p50_unidade_rpc": df_sol["prioritizationFee"].quantile(0.50),
        "taxa_p90_unidade_rpc": df_sol["prioritizationFee"].quantile(0.90),
        "taxa_maxima_unidade_rpc": df_sol["prioritizationFee"].max(),
        "proporcao_taxa_zero": (
            df_sol["prioritizationFee"].eq(0).mean()
        )
    }

    df_sol_resumo = salvar_csv(
        [resumo_sol], "solana_onchain_resumo.csv"
    )

    display(df_sol_resumo)
else:
    print("A API não retornou observações de taxas.")

Arquivo salvo: dados_cripto/solana_onchain_resumo.csv | Registros: 1


,timestamp_coleta_utc,observacoes,taxa_p50_unidade_rpc,taxa_p90_unidade_rpc,taxa_maxima_unidade_rpc,proporcao_taxa_zero
0,2026-11-02T20:01:49.433719+00:00,150,0.0,0.0,0,1.0


In [12]:
df_fng = pd.DataFrame(dados_fng)

df_fng["timestamp_utc"] = pd.to_datetime(
    pd.to_numeric(df_fng["timestamp"]),
    unit="s",
    utc=True
)

df_fng["value"] = pd.to_numeric(
    df_fng["value"], errors="coerce"
)

df_fng.to_csv(
    PASTA_DADOS / "fear_greed_historico.csv",
    index=False
)

display(
    df_fng[["timestamp_utc", "value", "value_classification"]]
)

,timestamp_utc,value,value_classification
0,2026-11-02 00:00:00+00:00,37,Fear
1,2026-11-01 00:00:00+00:00,34,Fear
2,2026-10-31 00:00:00+00:00,30,Fear
3,2026-10-30 00:00:00+00:00,30,Fear
4,2026-10-29 00:00:00+00:00,51,Neutral
5,2026-10-28 00:00:00+00:00,51,Neutral
6,2026-10-27 00:00:00+00:00,48,Neutral
7,2026-10-26 00:00:00+00:00,50,Neutral
8,2026-10-25 00:00:00+00:00,54,Neutral
9,2026-10-24 00:00:00+00:00,51,Neutral


In [13]:
print("=== DIAGNÓSTICO DA COLETA ===\n")

for arquivo in sorted(PASTA_DADOS.glob("*.csv")):
    df = pd.read_csv(arquivo)

    print(f"Arquivo: {arquivo.name}")
    print(f"Linhas: {len(df)}")
    print(f"Colunas: {list(df.columns)}")
    print("-" * 60)

print("\nArquivos disponíveis:")
for arquivo in sorted(PASTA_DADOS.iterdir()):
    print(arquivo.name, "-", arquivo.stat().st_size, "bytes")

=== DIAGNÓSTICO DA COLETA ===

Arquivo: bitcoin_onchain.csv
Linhas: 1
Colunas: ['timestamp_coleta_utc', 'mempool_transacoes', 'mempool_vsize_vbytes', 'mempool_taxas_sats', 'taxa_fastest_sat_vb', 'taxa_30min_sat_vb', 'taxa_1h_sat_vb', 'taxa_economica_sat_vb', 'taxa_minima_sat_vb']
------------------------------------------------------------
Arquivo: ethereum_onchain.csv
Linhas: 1
Colunas: ['timestamp_coleta_utc', 'bloco_atual', 'gas_price_gwei', 'base_fee_ultimo_bloco_gwei', 'base_fee_proximo_bloco_gwei', 'priority_fee_p50_gwei', 'gas_utilizado_ultimo_bloco']
------------------------------------------------------------
Arquivo: fear_greed_historico.csv
Linhas: 30
Colunas: ['value', 'value_classification', 'timestamp', 'time_until_update', 'timestamp_utc']
------------------------------------------------------------
Arquivo: mercado_horario.csv
Linhas: 144
Colunas: ['timestamp_abertura', 'abertura', 'maxima', 'minima', 'fechamento', 'volume', 'timestamp_fechamento', 'volume_quote', 'nume

In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("dados_cripto/mercado_horario.csv")

df["timestamp_abertura"] = pd.to_datetime(
    df["timestamp_abertura"], utc=True
)

df["timestamp_fechamento"] = pd.to_datetime(
    df["timestamp_fechamento"], utc=True
)

# Excluir o candle que ainda está em formação
agora = pd.Timestamp.now(tz="UTC")

df = df[df["timestamp_fechamento"] < agora].copy()

df = df.sort_values(["ativo", "timestamp_abertura"])

# Retorno percentual e logarítmico
df["retorno_pct"] = (
    df.groupby("ativo")["fechamento"]
    .pct_change() * 100
)

df["retorno_log"] = (
    df.groupby("ativo")["fechamento"]
    .transform(lambda s: np.log(s / s.shift(1)))
)

# Volatilidade realizada aproximada em janela de 24 horas
df["volatilidade_24h"] = (
    df.groupby("ativo")["retorno_log"]
    .transform(lambda s: s.rolling(24).std() * np.sqrt(24))
)

print("Candles válidos por ativo:")
print(df.groupby("ativo").size())

display(
    df[
        [
            "timestamp_abertura",
            "ativo",
            "fechamento",
            "retorno_pct",
            "volatilidade_24h"
        ]
    ].tail(12)
)

Candles válidos por ativo:
ativo
BTC    48
ETH    48
SOL    48
dtype: int64


,timestamp_abertura,ativo,fechamento,retorno_pct,volatilidade_24h
132,2026-11-02 08:00:00+00:00,SOL,94.64,0.381841,0.021092
133,2026-11-02 09:00:00+00:00,SOL,94.34,-0.316991,0.020856
134,2026-11-02 10:00:00+00:00,SOL,94.19,-0.158999,0.020882
135,2026-11-02 11:00:00+00:00,SOL,94.09,-0.106168,0.020876
136,2026-11-02 12:00:00+00:00,SOL,94.09,0.000000,0.020438
137,2026-11-02 13:00:00+00:00,SOL,94.35,0.276331,0.020630
138,2026-11-02 14:00:00+00:00,SOL,94.39,0.042395,0.020626
139,2026-11-02 15:00:00+00:00,SOL,93.55,-0.889925,0.022286
140,2026-11-02 16:00:00+00:00,SOL,93.41,-0.149653,0.022298
141,2026-11-02 17:00:00+00:00,SOL,92.20,-1.295365,0.025524


In [16]:
df = pd.read_csv("dados_cripto/mercado_horario.csv")

# Converter datas
df["timestamp_abertura"] = pd.to_datetime(
    df["timestamp_abertura"], utc=True
)
df["timestamp_fechamento"] = pd.to_datetime(
    df["timestamp_fechamento"], utc=True
)

# Converter campos numéricos
colunas_numericas = [
    "abertura", "maxima", "minima", "fechamento", "volume"
]

for coluna in colunas_numericas:
    if coluna in df.columns:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

# Resumo por ativo
print("Quantidade de registros:")
print(df.groupby("ativo").size())

print("\nValores ausentes:")
print(df.isna().sum())

print("\nCandles duplicados:")
print(
    df.duplicated(
        subset=["ativo", "timestamp_abertura"]
    ).sum()
)

print("\nVerificação dos preços:")
print("Fechamento menor ou igual a zero:",
      (df["fechamento"] <= 0).sum())

print("Máxima menor que mínima:",
      (df["maxima"] < df["minima"]).sum()
      if {"maxima", "minima"}.issubset(df.columns)
      else "Colunas não encontradas")

print("\nIntervalo temporal por ativo:")
print(
    df.groupby("ativo")["timestamp_abertura"]
      .agg(["min", "max"])
)

Quantidade de registros:
ativo
BTC    48
ETH    48
SOL    48
dtype: int64

Valores ausentes:
timestamp_abertura      0
abertura                0
maxima                  0
minima                  0
fechamento              0
volume                  0
timestamp_fechamento    0
volume_quote            0
numero_trades           0
volume_taker_base       0
volume_taker_quote      0
campo_ignorado          0
ativo                   0
fonte                   0
dtype: int64

Candles duplicados:
0

Verificação dos preços:
Fechamento menor ou igual a zero: 0
Máxima menor que mínima: 0

Intervalo temporal por ativo:
                            min                       max
ativo                                                    
BTC   2026-10-31 20:00:00+00:00 2026-11-02 19:00:00+00:00
ETH   2026-10-31 20:00:00+00:00 2026-11-02 19:00:00+00:00
SOL   2026-10-31 20:00:00+00:00 2026-11-02 19:00:00+00:00


In [3]:
import pandas as pd

df = pd.read_csv("dados_cripto/mercado_horario.csv")

print(df.groupby("ativo").size())

print("\nPeríodo coletado:")
print(
    df.groupby("ativo")["timestamp_abertura"]
      .agg(["min", "max"])
)

print("\nTotal de registros:", len(df))

ativo
BTC    999
ETH    999
SOL    999
dtype: int64

Período coletado:
                             min                        max
ativo                                                      
BTC    2026-09-22 05:00:00+00:00  2026-11-02 19:00:00+00:00
ETH    2026-09-22 05:00:00+00:00  2026-11-02 19:00:00+00:00
SOL    2026-09-22 05:00:00+00:00  2026-11-02 19:00:00+00:00

Total de registros: 2997


In [5]:
import pandas as pd
from pathlib import Path

arquivos = [
    "bitcoin_onchain.csv",
    "ethereum_onchain.csv",
    "solana_onchain_resumo.csv",
    "solana_taxas_prioridade.csv",
    "fear_greed_historico.csv"
]

for arquivo in arquivos:

    caminho = Path("./dados_cripto") / arquivo

    print("\n" + "=" * 60)
    print(arquivo)

    if caminho.exists():

        df = pd.read_csv(caminho)

        print("Registros:", len(df))
        print("Colunas:", df.columns.tolist())

        display(df.head(3))

    else:
        print("Arquivo não encontrado.")


bitcoin_onchain.csv
Registros: 1
Colunas: ['timestamp_coleta_utc', 'mempool_transacoes', 'mempool_vsize_vbytes', 'mempool_taxas_sats', 'taxa_fastest_sat_vb', 'taxa_30min_sat_vb', 'taxa_1h_sat_vb', 'taxa_economica_sat_vb', 'taxa_minima_sat_vb']


,timestamp_coleta_utc,mempool_transacoes,mempool_vsize_vbytes,mempool_taxas_sats,taxa_fastest_sat_vb,taxa_30min_sat_vb,taxa_1h_sat_vb,taxa_economica_sat_vb,taxa_minima_sat_vb
0,2026-11-02T20:01:19.910039+00:00,50178,22418931,5586189,2,1,1,1,1



ethereum_onchain.csv
Registros: 1
Colunas: ['timestamp_coleta_utc', 'bloco_atual', 'gas_price_gwei', 'base_fee_ultimo_bloco_gwei', 'base_fee_proximo_bloco_gwei', 'priority_fee_p50_gwei', 'gas_utilizado_ultimo_bloco']


,timestamp_coleta_utc,bloco_atual,gas_price_gwei,base_fee_ultimo_bloco_gwei,base_fee_proximo_bloco_gwei,priority_fee_p50_gwei,gas_utilizado_ultimo_bloco
0,2026-11-02T20:01:38.136070+00:00,26329337,0.201371,0.201369,0.20302,0.007331,0.53279



solana_onchain_resumo.csv
Registros: 1
Colunas: ['timestamp_coleta_utc', 'observacoes', 'taxa_p50_unidade_rpc', 'taxa_p90_unidade_rpc', 'taxa_maxima_unidade_rpc', 'proporcao_taxa_zero']


,timestamp_coleta_utc,observacoes,taxa_p50_unidade_rpc,taxa_p90_unidade_rpc,taxa_maxima_unidade_rpc,proporcao_taxa_zero
0,2026-11-02T20:01:49.433719+00:00,150,0.0,0.0,0,1.0



solana_taxas_prioridade.csv
Registros: 150
Colunas: ['prioritizationFee', 'slot', 'timestamp_coleta_utc']


,prioritizationFee,slot,timestamp_coleta_utc
0,0,464621226,2026-11-02T20:01:49.433719+00:00
1,0,464621227,2026-11-02T20:01:49.433719+00:00
2,0,464621228,2026-11-02T20:01:49.433719+00:00



fear_greed_historico.csv
Registros: 30
Colunas: ['value', 'value_classification', 'timestamp', 'time_until_update', 'timestamp_utc']


,value,value_classification,timestamp,time_until_update,timestamp_utc
0,37,Fear,1793577600,14434.0,2026-11-02 00:00:00+00:00
1,34,Fear,1793491200,NaN,2026-11-01 00:00:00+00:00
2,30,Fear,1793404800,NaN,2026-10-31 00:00:00+00:00


## 6. Coleta Histórica de 2 Anos: Candles de 5 Minutos (Binance Spot)

Esta célula estende a coleta de mercado para cobrir **2 anos completos (17.520 horas)** com candles de **5 minutos** para Bitcoin () e Ethereum ().
Os retornos de 5 minutos são a base para o cálculo da **Volatilidade Realizada Horária ($)**, permitindo compatibilidade total com as séries on-chain de 2 anos do BigQuery.

In [ ]:
import time
from datetime import datetime, timezone
import requests
import pandas as pd
import numpy as np
from pathlib import Path

# Definir período de 2 anos alinhado com BigQuery (02/11/2024 a 02/11/2026)
PASTA_2ANOS = Path("./dados_cripto/historico_2anos")
PASTA_2ANOS.mkdir(parents=True, exist_ok=True)
ARQ_5M_2A = PASTA_2ANOS / "mercado_5m.csv"

INICIO_2A = datetime(2024, 11, 2, 23, 0, 0, tzinfo=timezone.utc)
FIM_2A = datetime(2026, 11, 2, 23, 0, 0, tzinfo=timezone.utc)
SIMBOLOS_2A = {"BTC": "BTCUSDT", "ETH": "ETHUSDT"}
COLS_KLINE = [
    "abertura_ts", "abertura", "maxima", "minima", "fechamento", "volume",
    "fechamento_ts", "volume_quote", "numero_trades", "volume_taker_base", "volume_taker_quote"
]
BASES_BINANCE = ["https://api.binance.com", "https://data-api.binance.vision"]

def klines_paginado(simbolo, intervalo, inicio_dt, fim_dt):
    ini_ms = int(inicio_dt.timestamp() * 1000)
    fim_ms = int(fim_dt.timestamp() * 1000)
    registros = []
    print(f"Iniciando coleta {simbolo} ({intervalo}) de {inicio_dt} até {fim_dt}...")
    while ini_ms < fim_ms:
        dados = None
        for base in BASES_BINANCE:
            try:
                resp = requests.get(
                    f"{base}/api/v3/klines",
                    params={"symbol": simbolo, "interval": intervalo, "startTime": ini_ms, "endTime": fim_ms - 1, "limit": 1000},
                    timeout=15
                )
                if resp.status_code == 200:
                    dados = resp.json()
                    break
            except Exception:
                time.sleep(0.5)
        if not dados:
            break
        registros += [d[:11] for d in dados]
        ini_ms = dados[-1][0] + 1
        if len(dados) < 1000:
            break
        time.sleep(0.1)
    return registros

# Executar coleta para BTC e ETH
dfs_5m = []
for ativo, simb in SIMBOLOS_2A.items():
    k = klines_paginado(simb, "5m", INICIO_2A, FIM_2A)
    df_ativo = pd.DataFrame(k, columns=COLS_KLINE)
    df_ativo["abertura_ts"] = pd.to_datetime(df_ativo["abertura_ts"], unit="ms", utc=True)
    df_ativo["fechamento_ts"] = pd.to_datetime(df_ativo["fechamento_ts"], unit="ms", utc=True)
    for col in [c for c in COLS_KLINE if "ts" not in c]:
        df_ativo[col] = pd.to_numeric(df_ativo[col], errors="coerce")
    df_ativo["ativo"] = ativo
    dfs_5m.append(df_ativo)
    print(f"{ativo}: {len(df_ativo):,} candles de 5m coletados.")

df_5m_2anos = pd.concat(dfs_5m, ignore_index=True).drop_duplicates(["ativo", "abertura_ts"]).sort_values(["ativo", "abertura_ts"]).reset_index(drop=True)
df_5m_2anos.to_csv(ARQ_5M_2A, index=False)
print(f"Arquivo salvo com sucesso em: {ARQ_5M_2A} | Total: {len(df_5m_2anos):,} candles.")
display(df_5m_2anos.groupby("ativo")["abertura_ts"].agg(["min", "max", "count"]))
